# NLP Summary Generation: TF-IDF, Encoder-Decoder, and Bahdanau Attention

**Authors:** Skander Adam Afi, Mohamed Harzallah

This notebook implements and **quantitatively compares** three approaches to
automatic text summarization on the **CNN / DailyMail** dataset:

1. **TF-IDF extractive summarization** (rank & pick existing sentences)
2. **RNN Encoder-Decoder (seq2seq)** abstractive summarizer (LSTM, teacher forcing)
3. **RNN Encoder-Decoder with Bahdanau Attention**

The dataset ships as three files (`train.csv`, `validation.csv`, `test.csv`),
each with an `article` column (full text) and a `highlights` column, which we use
as the **reference summary**. All three models are evaluated with **ROUGE-1/2/L**
on the test split so the comparison is measurable rather than qualitative.


## 0. Setup

In [ ]:
# Install dependencies (run once, e.g. on Google Colab)
%pip install -q tensorflow rouge-score nltk scikit-learn pandas


In [ ]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import nltk
from nltk.tokenize import sent_tokenize

# NLTK data needed for sentence tokenization
nltk.download("stopwords")
nltk.download("punkt")
try:
    nltk.download("punkt_tab")   # required by newer NLTK versions
except Exception:
    pass

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


In [ ]:
# ---------------- Configuration ----------------
# CSV files live in the data/ folder next to this notebook.
# On Google Colab, upload them and set DATA_DIR = "/content/".
DATA_DIR    = "data/"
TRAIN_CSV   = DATA_DIR + "train.csv"
VAL_CSV     = DATA_DIR + "validation.csv"
TEST_CSV    = DATA_DIR + "test.csv"

TEXT_COL    = "article"      # full article
SUMMARY_COL = "highlights"   # reference summary

# train.csv is large (~1.2 GB / ~287k rows): read only the first N rows.
TRAIN_SIZE      = 20000   # rows read from train.csv      (set None to use all)
VAL_SIZE        = 2000    # rows read from validation.csv
TEST_SIZE       = 2000    # rows read from test.csv
EVAL_SIZE       = 300     # examples used for ROUGE (greedy decoding is slow)

MAX_TEXT_LEN    = 150     # max tokens kept from each article
MAX_SUMMARY_LEN = 40      # max tokens kept from each summary
VOCAB_SIZE      = 30000   # max vocabulary size (shared encoder/decoder vocab)
EMBEDDING_DIM   = 100
LATENT_DIM      = 128
BATCH_SIZE      = 64
EPOCHS          = 15      # increase for better results if you have a GPU


In [ ]:
# ---------------- Load & clean the three splits ----------------
def clean_text(text):
    """Lowercase, strip HTML, keep letters only, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)     # keep letters and spaces
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text

def load_split(path, n_rows):
    """Read a CNN/DailyMail csv split, clean it, drop empties."""
    df = pd.read_csv(path, nrows=n_rows, usecols=[TEXT_COL, SUMMARY_COL])
    df = df.dropna(subset=[TEXT_COL, SUMMARY_COL]).reset_index(drop=True)
    df["clean_body"]    = df[TEXT_COL].apply(clean_text)
    df["clean_summary"] = df[SUMMARY_COL].apply(clean_text)
    df = df[(df["clean_body"].str.len() > 0) &
            (df["clean_summary"].str.len() > 0)].reset_index(drop=True)
    return df

df_train = load_split(TRAIN_CSV, TRAIN_SIZE)
df_val   = load_split(VAL_CSV,   VAL_SIZE)
df_test  = load_split(TEST_CSV,  TEST_SIZE)

print("Train:", len(df_train), "| Validation:", len(df_val), "| Test:", len(df_test))
df_train[["clean_body", "clean_summary"]].head()


## 1. TF-IDF Extractive Summarization

A simple **extractive** baseline: split the article into sentences, score each
sentence by its **TF-IDF** weight, and keep the top-N highest-scoring sentences
(in their original order). Unlike a raw word-frequency count, TF-IDF down-weights
words that appear in many sentences, so generic terms contribute less.

Note: sentence splitting is done on the **raw** article text (which still has
punctuation), not on `clean_body`.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_extractive_summary(text, num_sentences=3):
    """Return the top-N sentences ranked by mean TF-IDF weight."""
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text.strip()

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform(sentences)          # (n_sentences, n_terms)

    # score = mean TF-IDF over the words present in the sentence
    sums    = np.asarray(tfidf.sum(axis=1)).ravel()
    lengths = np.asarray((tfidf > 0).sum(axis=1)).ravel()
    scores  = np.divide(sums, lengths, out=np.zeros_like(sums), where=lengths > 0)

    top_idx = np.argsort(scores)[::-1][:num_sentences]
    top_idx = sorted(top_idx)                            # preserve reading order
    return " ".join(sentences[i] for i in top_idx)

# Example (use the raw article so sentence splitting has punctuation)
example = df_test[TEXT_COL].iloc[0]
print("=== ORIGINAL ARTICLE (first 600 chars) ===")
print(example[:600], "...\n")
print("=== TF-IDF EXTRACTIVE SUMMARY ===")
print(tfidf_extractive_summary(example, num_sentences=3))


In [ ]:
# ---------------- ROUGE evaluation helper (reused by all models) ----------------
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate_rouge(references, predictions):
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    n = max(len(references), 1)
    for ref, pred in zip(references, predictions):
        s = scorer.score(ref, pred)
        for k in agg:
            agg[k] += s[k].fmeasure
    return {k: round(v / n, 4) for k, v in agg.items()}

# Evaluate TF-IDF on the test split (extraction runs on the raw article text)
eval_df      = df_test.head(EVAL_SIZE)
# clean the extracted sentences the same way as the references, so all models
# are scored on identical preprocessing (lowercase, letters only)
tfidf_preds  = [clean_text(tfidf_extractive_summary(t, 3)) for t in eval_df[TEXT_COL]]
tfidf_refs   = eval_df["clean_summary"].tolist()
tfidf_scores = evaluate_rouge(tfidf_refs, tfidf_preds)
print("TF-IDF ROUGE:", tfidf_scores)


## 2. RNN Encoder-Decoder (seq2seq)

An **abstractive** model that generates the summary token by token.

Setup:
- a **single shared tokenizer** fit on the **training** articles + summaries only
  (the validation/test splits are never used to build the vocabulary);
- every target summary is wrapped with `startseq ... endseq` markers;
- the official train / validation / test splits are used as-is;
- the model is trained with **teacher forcing** before inference;
- padding positions are excluded from the loss via `sample_weight`;
- greedy decoding never emits padding, `<unk>` or `startseq`.


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# add start/end markers to every target summary
for d in (df_train, df_val, df_test):
    d["dec_summary"] = "startseq " + d["clean_summary"] + " endseq"

# ONE tokenizer, fit on TRAINING data only -> shared vocabulary
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")
tokenizer.fit_on_texts(df_train["clean_body"].tolist() + df_train["dec_summary"].tolist())

start_token = tokenizer.word_index["startseq"]
end_token   = tokenizer.word_index["endseq"]
reverse_word_index = {i: w for w, i in tokenizer.word_index.items()}
print("Words in tokenizer:", len(tokenizer.word_index), "| Using vocab size:", VOCAB_SIZE)

def encode_split(d):
    X = pad_sequences(tokenizer.texts_to_sequences(d["clean_body"].tolist()),
                      maxlen=MAX_TEXT_LEN, padding="post", truncating="post")
    Y = pad_sequences(tokenizer.texts_to_sequences(d["dec_summary"].tolist()),
                      maxlen=MAX_SUMMARY_LEN, padding="post", truncating="post")
    return X, Y

X_train, Y_train = encode_split(df_train)
X_val,   Y_val   = encode_split(df_val)
X_test,  Y_test  = encode_split(df_test)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# decoder input = all tokens but the last ; target = all tokens but the first
def make_decoder_io(Yarr):
    return Yarr[:, :-1], np.expand_dims(Yarr[:, 1:], -1)

dec_in_train, dec_tgt_train = make_decoder_io(Y_train)
dec_in_val,   dec_tgt_val   = make_decoder_io(Y_val)

# padding positions (token 0) must not count in the loss / accuracy
w_train = (dec_tgt_train[..., 0] != 0).astype("float32")
w_val   = (dec_tgt_val[..., 0]   != 0).astype("float32")

# tokens the decoder must never emit at inference time
BANNED_TOKENS = [0, tokenizer.word_index["<unk>"], start_token]


In [ ]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.models import Model

VOCAB = VOCAB_SIZE

# ----- Encoder -----
encoder_inputs = Input(shape=(MAX_TEXT_LEN,), name="encoder_inputs")
enc_emb = Embedding(VOCAB, EMBEDDING_DIM, mask_zero=True, name="enc_embedding")(encoder_inputs)
encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# ----- Decoder -----
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
dec_emb_layer = Embedding(VOCAB, EMBEDDING_DIM, mask_zero=True, name="dec_embedding")
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = TimeDistributed(Dense(VOCAB, activation="softmax"), name="output_dense")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              weighted_metrics=["accuracy"])
model.summary()


In [ ]:
# ----- Train the seq2seq model -----
history = model.fit(
    [X_train, dec_in_train], dec_tgt_train,
    sample_weight=w_train,
    validation_data=([X_val, dec_in_val], dec_tgt_val, w_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
)


In [ ]:
# ----- Inference models (reuse trained layers) -----
encoder_model = Model(encoder_inputs, encoder_states)

dec_state_h = Input(shape=(LATENT_DIM,), name="dec_h")
dec_state_c = Input(shape=(LATENT_DIM,), name="dec_c")
dec_states_in = [dec_state_h, dec_state_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
dec_out2, h2, c2 = decoder_lstm(dec_emb2, initial_state=dec_states_in)
dec_out2 = decoder_dense(dec_out2)
decoder_model = Model([decoder_inputs] + dec_states_in, [dec_out2, h2, c2])

def pick_next_token(probs):
    """Greedy choice, never emitting padding / <unk> / startseq."""
    probs = np.array(probs, copy=True)
    probs[BANNED_TOKENS] = -1.0
    return int(np.argmax(probs))

def decode_sequence(input_seq):
    """Greedy decoding for the plain encoder-decoder."""
    # direct model calls are much faster than .predict() inside a loop
    states = [s.numpy() for s in encoder_model(input_seq, training=False)]
    target = np.array([[start_token]])
    words = []
    for _ in range(MAX_SUMMARY_LEN):
        out, h, c = decoder_model([target] + states, training=False)
        idx = pick_next_token(out.numpy()[0, -1, :])
        word = reverse_word_index.get(idx, "")
        if word == "endseq" or word == "":
            break
        words.append(word)
        target = np.array([[idx]])
        states = [h.numpy(), c.numpy()]
    return " ".join(words)


In [ ]:
# ----- Evaluate the seq2seq model with ROUGE on the test split -----
eval_n = min(EVAL_SIZE, len(df_test))
refs   = df_test["clean_summary"].iloc[:eval_n].tolist()
preds  = [decode_sequence(X_test[i].reshape(1, -1)) for i in range(eval_n)]
seq2seq_scores = evaluate_rouge(refs, preds)
print("Seq2Seq ROUGE:", seq2seq_scores)

# show one example
print("\nArticle (clean, first 300 chars):", df_test["clean_body"].iloc[0][:300])
print("Reference summary:", df_test["clean_summary"].iloc[0])
print("Generated summary:", decode_sequence(X_test[0].reshape(1, -1)))


## 3. RNN Encoder-Decoder with Bahdanau Attention

A plain encoder-decoder squeezes the whole article into a single fixed vector,
which loses information on long inputs. **Bahdanau attention** (Bahdanau,
Cho & Bengio, 2015, *Neural Machine Translation by Jointly Learning to Align and
Translate*) lets the decoder look back at *all* encoder hidden states at every
step and focus on the most relevant ones via a learned, softmax-weighted context
vector.

The encoder now returns its full sequence of hidden states (`return_sequences=True`)
so attention can be computed over them.


In [ ]:
from tensorflow.keras.layers import Concatenate, Layer

class BahdanauAttention(Layer):
    """Additive (Bahdanau) attention computed over all decoder timesteps at once."""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.W1 = Dense(units)   # projects encoder outputs
        self.W2 = Dense(units)   # projects decoder outputs
        self.V  = Dense(1)       # scoring layer

    def call(self, enc_output, dec_output):
        # enc_output: (batch, Tx, h) ; dec_output: (batch, Ty, h)
        enc = tf.expand_dims(self.W1(enc_output), 1)    # (batch, 1,  Tx, u)
        dec = tf.expand_dims(self.W2(dec_output), 2)    # (batch, Ty, 1,  u)
        score   = self.V(tf.nn.tanh(enc + dec))         # (batch, Ty, Tx, 1)
        score   = tf.squeeze(score, -1)                 # (batch, Ty, Tx)
        weights = tf.nn.softmax(score, axis=-1)         # (batch, Ty, Tx)
        context = tf.matmul(weights, enc_output)        # (batch, Ty, h)
        return context, weights


In [ ]:
# ----- Training model with attention -----
# Encoder returns the full hidden-state sequence
a_enc_inputs = Input(shape=(MAX_TEXT_LEN,), name="a_enc_inputs")
a_enc_emb = Embedding(VOCAB, EMBEDDING_DIM, name="a_enc_emb")(a_enc_inputs)
a_encoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="a_encoder_lstm")
a_enc_outputs, a_state_h, a_state_c = a_encoder_lstm(a_enc_emb)

# Decoder
a_dec_inputs = Input(shape=(None,), name="a_dec_inputs")
a_dec_emb_layer = Embedding(VOCAB, EMBEDDING_DIM, name="a_dec_emb")
a_dec_emb = a_dec_emb_layer(a_dec_inputs)
a_decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="a_decoder_lstm")
a_dec_outputs, _, _ = a_decoder_lstm(a_dec_emb, initial_state=[a_state_h, a_state_c])

attention = BahdanauAttention(LATENT_DIM, name="bahdanau_attention")
context, _ = attention(a_enc_outputs, a_dec_outputs)
concat = Concatenate(axis=-1)([a_dec_outputs, context])
a_decoder_dense = TimeDistributed(Dense(VOCAB, activation="softmax"), name="a_output_dense")
a_outputs = a_decoder_dense(concat)

attn_model = Model([a_enc_inputs, a_dec_inputs], a_outputs)
attn_model.compile(optimizer="rmsprop",
                   loss="sparse_categorical_crossentropy",
                   weighted_metrics=["accuracy"])
attn_model.summary()


In [ ]:
# ----- Train the attention model -----
attn_history = attn_model.fit(
    [X_train, dec_in_train], dec_tgt_train,
    sample_weight=w_train,
    validation_data=([X_val, dec_in_val], dec_tgt_val, w_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
)


In [ ]:
# ----- Attention inference models (reuse trained layers) -----
# Encoder inference: returns the full encoder sequence + final states
a_encoder_model = Model(a_enc_inputs, [a_enc_outputs, a_state_h, a_state_c])

# Single-step decoder
a_dec_in_single = Input(shape=(1,), name="a_dec_in_single")
a_enc_out_in    = Input(shape=(MAX_TEXT_LEN, LATENT_DIM), name="a_enc_out_in")
a_h_in          = Input(shape=(LATENT_DIM,), name="a_h_in")
a_c_in          = Input(shape=(LATENT_DIM,), name="a_c_in")

a_dec_emb_inf = a_dec_emb_layer(a_dec_in_single)
a_dec_out_inf, a_h, a_c = a_decoder_lstm(a_dec_emb_inf, initial_state=[a_h_in, a_c_in])
a_context_inf, a_weights_inf = attention(a_enc_out_in, a_dec_out_inf)
a_concat_inf = Concatenate(axis=-1)([a_dec_out_inf, a_context_inf])
a_out_inf = a_decoder_dense(a_concat_inf)

a_decoder_model = Model([a_dec_in_single, a_enc_out_in, a_h_in, a_c_in],
                        [a_out_inf, a_h, a_c])

def decode_sequence_attn(input_seq):
    """Greedy decoding for the attention model."""
    enc_out, h, c = [t.numpy() for t in a_encoder_model(input_seq, training=False)]
    target = np.array([[start_token]])
    words = []
    for _ in range(MAX_SUMMARY_LEN):
        out, h, c = a_decoder_model([target, enc_out, h, c], training=False)
        idx = pick_next_token(out.numpy()[0, -1, :])
        word = reverse_word_index.get(idx, "")
        if word == "endseq" or word == "":
            break
        words.append(word)
        target = np.array([[idx]])
        h, c = h.numpy(), c.numpy()
    return " ".join(words)


In [ ]:
# ----- Evaluate the attention model with ROUGE on the test split -----
attn_preds  = [decode_sequence_attn(X_test[i].reshape(1, -1)) for i in range(eval_n)]
attn_scores = evaluate_rouge(refs, attn_preds)
print("Attention ROUGE:", attn_scores)

print("\nReference summary:", df_test["clean_summary"].iloc[0])
print("Generated (attention):", decode_sequence_attn(X_test[0].reshape(1, -1)))


## 4. Results comparison

Final ROUGE F-measures for the three techniques on the test split.


In [ ]:
results = pd.DataFrame([
    {"Model": "TF-IDF (extractive)",        **tfidf_scores},
    {"Model": "RNN Encoder-Decoder",        **seq2seq_scores},
    {"Model": "RNN + Bahdanau Attention",   **attn_scores},
])
results = results[["Model", "rouge1", "rouge2", "rougeL"]]
results


### Notes & limitations

- CNN/DailyMail articles are long (hundreds of tokens) and the from-scratch
  neural models are trained on a capped subset (`TRAIN_SIZE`) with a limited
  vocabulary and few epochs, so their absolute ROUGE scores are modest. Increase
  `TRAIN_SIZE`, `VOCAB_SIZE`, `MAX_TEXT_LEN`, and `EPOCHS` (ideally on a GPU) for
  stronger results.
- The extractive TF-IDF baseline is typically a hard baseline to beat on
  CNN/DailyMail, since the highlights are largely extractive in nature.
- Padding is ignored in the training loss, but the attention scores do not mask
  padded encoder positions (rare here, since most articles exceed `MAX_TEXT_LEN`); adding a padding
  mask would be a natural next improvement.
- For a state-of-the-art baseline, compare against a pretrained model such as
  `facebook/bart-large-cnn` or `t5-small` via Hugging Face Transformers.
